In [ ]:
import os
import warnings
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

import tensorflow as tf

tf.get_logger().setLevel('ERROR')
tf.get_logger().setLevel("ERROR")
warnings.simplefilter("ignore")


In [ ]:
import tensorflow as tf
# import tensorflow_addons as tfa
from tensorflow import keras
import keras_tuner as kt
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GlobalAveragePooling2D, multiply, Conv2D, Dense, BatchNormalization, Input, MaxPool2D, Dropout, Flatten
import numpy as np
from sklearn.preprocessing import normalize
from keras import layers
from keras import ops
import keras



AUTO = tf.data.AUTOTUNE

def get_intial_fold_dataset(training_config, path, seed, shuffle, cache = False):
  if not training_config["ENABLE_DETERMINISM"]:
    seed = None

  dataset = tf.data.TFRecordDataset(tf.io.gfile.glob(path + "/*.tfrec"), num_parallel_reads=AUTO) # if TPU else 20)

  if(shuffle):
    dataset = dataset.shuffle(training_config["SHUFFLE_BUFFER"], seed = seed)

  dataset = dataset.map(lambda records: tf.io.parse_single_example(
      records,
      {
          "image": tf.io.FixedLenFeature([], dtype=tf.string),
          "class": tf.io.FixedLenFeature([], dtype=tf.int64),

          # "label": tf.io.FixedLenFeature([], dtype=tf.string),
          "objid": tf.io.FixedLenFeature([], dtype=tf.string),
          # "one_hot_class": tf.io.VarLenFeature(tf.float32)
      }),
      num_parallel_calls=AUTO)
  dataset = dataset.map(lambda item: (tf.reshape(tf.image.decode_jpeg(item['image'], channels=3), [training_config["IMAGE_SIZE"], training_config["IMAGE_SIZE"], 3]), item['class']), num_parallel_calls=AUTO)
  if(cache):
    print(f"caching jpeg dataset into memory")
    dataset = dataset.cache()

  dataset = dataset.map(lambda x,y: (tf.cast(x, tf.float32), y), num_parallel_calls=AUTO)
  dataset = dataset.map(lambda x,y: (tf.keras.layers.Rescaling(scale=1./255)(x), y), num_parallel_calls=AUTO)

  return dataset

training_config = {
"ENABLE_DETERMINISM": False,
"SEED": 17,
"AUTO": tf.data.AUTOTUNE,
"TRAIN_BATCH_SIZE": 64,
"TEST_BATCH_SIZE": 64,
"NUMBER_OF_EPOCHS": 500,
"EARLY_STOPPING_TOLERANCE": 10,
"LEARNING_RATE": 1e-4,
"IMAGE_SIZE": 128,
"SHUFFLE_BUFFER": 4096,
"TPU": False,
"AUGMENTATIONS_ROTATE": False,
"AUGMENTATIONS_FLIP_HORIZONTALLY": False,
"AUGMENTATIONS_FLIP_VERTICALLY": False,
"AUGMENTATIONS_ZOOM": False,
"AUGMENTATIONS_RANDOM_NOISE": False,
"AUGMENTATIONS_CENTER_NOISE": False,
"AUGMENTATIONS_OUTSIDE_NOISE": False,
"AUGMENTATIONS_PERLIN_CENTER_NOISE": False,
"USE_ADABELIEF_OPTIMIZER": False,
"GALAXY_DATASET": "SDSS",
"TRAIN_DATASET_SIZE": 2 * 4096 + 2048,
"TEST_DATASET_SIZE": 1025,
"STRETCH": False,
"STEPS_PER_EXECUTION": 1,
"FOLDS": 1,
"NUMBER_OF_CLASSES": 11
}
training_config["WANDB_PROJECT_NAME"] = f"TrainTestValidation, {training_config['NUMBER_OF_CLASSES']} classes"
training_config["EXPERIMENT_DESCRIPTION"] = f"asd"
training_config["LOCAL_GCP_PATH_BASE"] = f"TFDataset/11Classes"
training_config["REMOTE_GCP_PATH_BASE"] = f"TFDataset/11Classes"
training_config["DATASET_PATH"] = f"tuning"


@keras.saving.register_keras_serializable()
class FocalConv(keras.layers.Layer):
    def __init__(self, filters, kernel_size, factor, activation):
        super().__init__()
        self.filters = filters
        self.kernel = kernel_size
        self.activation = activation
        self.factor = factor
        self.conv1 = Conv2D(filters=self.filters, kernel_size=self.kernel, activation=self.activation, padding='same')

    def build(self, input_shape):
        self.gauss = self.make_gaussian(size=input_shape[1], depth=self.filters, factor=self.factor)

    def call(self, x, training=False):
      x = self.conv1(x, training=training)
      batch_size = tf.shape(x)[0]
      xx = tf.repeat(self.gauss, repeats=batch_size, axis=0)
      x = multiply([x, xx])

      return x
    
    def get_config(self):
       return {
          "filters": self.filters,
          "kernel_size": self.kernel,
          "activation": self.activation,
          "factor": self.factor
       }

    def make_gaussian(self, size: int, depth: int, factor: float):
        """ Make a square gaussian kernel.

        size is the length of a side of the square
        fwhm is full-width-half-maximum, which
        can be thought of as an effective radius.
        """
        fwhm = size // 2
        x = np.arange(0, size, 1, float)
        y = x[:,np.newaxis]

        x0 = y0 = size // 2

        gauss = np.exp(-1 * factor * np.log(2) * ((x-x0) ** 2 + (y-y0) ** 2) / fwhm **2)

        return tf.expand_dims(np.stack([gauss for _ in range(depth)], -1), axis=0)


@keras.saving.register_keras_serializable()
class WaveConv(keras.layers.Layer):
    def __init__(self, filters, kernel_size, k_factor, activation):
        super().__init__()
        self.filters = filters
        self.kernel = kernel_size
        self.activation = activation
        self.k_factor = k_factor
        self.conv1 = Conv2D(filters=self.filters, kernel_size=self.kernel, activation=self.activation, padding='same')

    def build(self, input_shape):
        self.wave = self.make_wave(size=input_shape[1], depth=self.filters, k_factor=self.k_factor)

    def call(self, x, training=False):
      x = self.conv1(x, training=training)
      batch_size = tf.shape(x)[0]
      xx = tf.repeat(self.wave, repeats=batch_size, axis=0)
      x = multiply([x, xx])

      return x
    
    def get_config(self):
       return {
          "filters": self.filters,
          "kernel_size": self.kernel,
          "activation": self.activation,
          "k_factor": self.k_factor
       }

    def make_wave(self, size: int, depth: int, k_factor: float = 2):
        x = np.arange(0, size, 1, float)
        y = x[:,np.newaxis]
        x0 = y0 = size // 2

        k = np.pi / k_factor
        x, y = x - x0, y - y0
        l = np.sqrt(x * x + y * y)
        u = np.cos(k * l )
        u = normalize(u)
        return 1 - tf.expand_dims(np.stack([u for _ in range(depth)], -1), axis=0)


class SparseF1Score(tf.keras.metrics.F1Score):

    def __init__(self, name='f1_score', **kwargs):
        super().__init__(name=name, **kwargs)

    def update_state(self, y_true, y_pred, sample_weight=None):
        super().update_state(tf.reshape(tf.one_hot(y_true, training_config["NUMBER_OF_CLASSES"]), [-1, training_config["NUMBER_OF_CLASSES"]]), y_pred, sample_weight)

    def result(self):
        return super().result()

def model_builder(hp):
    
  model = Sequential(name="Custom")
  model.add(Input(shape=(128, 128, 3)))

  num_layers = hp.Int('num_layers', min_value=3, max_value=5, step=1)

  model = Sequential(name="Cavanagh")
  model.add(Input(shape=(training_config["IMAGE_SIZE"], training_config["IMAGE_SIZE"], 3)))

  for i in range(num_layers):

    layer_type = hp.Choice(f"type_{i}", ["SimpleConv", "FocalConv", "WaveConv"])
    focal_factor = hp.Int(f'focal_factor_{i}', parent_name=f'type_{i}', parent_values=["FocalConv"], min_value=1, max_value=10, step=1)
    wave_factor = hp.Int(f'wave_factor_{i}', parent_name=f'type_{i}', parent_values=["WaveConv"], min_value=1, max_value=10, step=1)
    kernel_size = hp.Int(f'kernel_size_{i}', min_value=3, max_value=7, step=2)
    filters = hp.Int(f'filters_{i}', min_value=16, max_value=64, step=16)

    if (layer_type == "SimpleConv"):
      model.add(Conv2D(filters=filters, kernel_size=kernel_size, activation='relu', padding='same'))
    elif(layer_type == "FocalConv"):
      model.add(FocalConv(filters=filters, kernel_size=kernel_size, activation='relu', factor= focal_factor))
    elif(layer_type == "WaveConv"):
      model.add(WaveConv(filters=filters, kernel_size=kernel_size, activation='relu', k_factor= wave_factor))

    model.add(BatchNormalization())
    model.add(MaxPool2D(2))


  

  model.add(Flatten())
  model.add(Dropout(0.5))

  num_dense = hp.Int('num_dense', min_value=1, max_value=3, step=1)
  for j in range(num_dense):
    dense_output = hp.Int(f'dense_output_{j}', min_value=32, max_value=256, step=32)
    model.add(Dense(dense_output, activation='relu'))
  
  model.add(Dense(training_config["NUMBER_OF_CLASSES"], activation='softmax'))

  model.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-4, beta_1=0.9, beta_2=0.999, epsilon=1e-08),
              loss=keras.losses.SparseCategoricalCrossentropy(),
              metrics=[tf.keras.metrics.SparseCategoricalAccuracy(), SparseF1Score(
                average='macro',
            )])
  
  return model


gpus = tf.config.list_physical_devices('GPU')
strategy = tf.distribute.MirroredStrategy()

test_dataset = get_intial_fold_dataset(
    training_config,
    f"{training_config['REMOTE_GCP_PATH_BASE']}/{training_config['DATASET_PATH']}/test",
    seed = training_config["SEED"],
    shuffle = False,
    cache = True).batch(training_config["TEST_BATCH_SIZE"])

train_dataset = get_intial_fold_dataset(
    training_config,
    f"{training_config['REMOTE_GCP_PATH_BASE']}/{training_config['DATASET_PATH']}/train",
    training_config["SEED"],
    shuffle = True,
    cache = True).batch(training_config["TEST_BATCH_SIZE"])

stop_early = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=10)

# train_dataset = train_dataset.repeat()
# test_dataset = test_dataset.repeat()

tuner = kt.Hyperband(
    model_builder,
    objective=kt.Objective("val_f1_score", direction="max"),
    max_epochs=100,
    overwrite=False,
    hyperband_iterations = 5,
    directory='tuning_models',
    project_name='best_arch_by_f1_no_augmentations',
    max_consecutive_failed_trials=1,
    distribution_strategy=strategy)

tuner.search(x= train_dataset,
            validation_data = test_dataset,
            epochs = 500,
            callbacks=[stop_early],
            # steps_per_epoch = (5 * 4096) // 128,
            # validation_steps = 2264 // 128,
            verbose = 1,
            shuffle = False)



In [ ]:
top_hyperparams = tuner.get_best_hyperparameters(1)

for hyperparams in top_hyperparams:
    for prop, val in hyperparams.values.items():
        if "tuner" in prop:
            continue

        print(f'{prop}: {val}')



In [ ]:
from collections import Counter

def Most_Common(lst):
    data = Counter(lst)
    return data.most_common(1)[0][0]

top_hyperparams = tuner.get_best_hyperparameters(10)

choices = {}

for hyperparams in top_hyperparams:
    for prop, val in hyperparams.values.items():
        if "tuner" in prop:
            continue

        if prop not in choices:
             choices[prop] = []

        choices[prop].append(val)


choices = dict(sorted(choices.items(), key= lambda c: c[-2:]))


for key in choices:
    choices[key] = sorted(choices[key])
    print(key, ":", Most_Common(choices[key]))



# Additional tuning experiments

In [ ]:
import tensorflow as tf
# import tensorflow_addons as tfa
from tensorflow import keras
import keras_tuner as kt
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GlobalAveragePooling2D, multiply, Conv2D, Dense, BatchNormalization, Input, MaxPool2D, Dropout, Flatten, Concatenate, add, AveragePooling2D
import numpy as np
from sklearn.preprocessing import normalize
from keras import layers
from keras import ops
import keras
from scipy.ndimage import rotate


AUTO = tf.data.AUTOTUNE

def get_intial_fold_dataset(training_config, path, seed, shuffle, cache = False):
  if not training_config["ENABLE_DETERMINISM"]:
    seed = None

  dataset = tf.data.TFRecordDataset(tf.io.gfile.glob(path + "/*.tfrec"), num_parallel_reads=AUTO) # if TPU else 20)

  if(shuffle):
    dataset = dataset.shuffle(training_config["SHUFFLE_BUFFER"], seed = seed)

  dataset = dataset.map(lambda records: tf.io.parse_single_example(
      records,
      {
          "image": tf.io.FixedLenFeature([], dtype=tf.string),
          "class": tf.io.FixedLenFeature([], dtype=tf.int64),

          # "label": tf.io.FixedLenFeature([], dtype=tf.string),
          "objid": tf.io.FixedLenFeature([], dtype=tf.string),
          # "one_hot_class": tf.io.VarLenFeature(tf.float32)
      }),
      num_parallel_calls=AUTO)
  dataset = dataset.map(lambda item: (tf.reshape(tf.image.decode_jpeg(item['image'], channels=3), [training_config["IMAGE_SIZE"], training_config["IMAGE_SIZE"], 3]), item['class']), num_parallel_calls=AUTO)
  if(cache):
    print(f"caching jpeg dataset into memory")
    dataset = dataset.cache()

  dataset = dataset.map(lambda x,y: (tf.cast(x, tf.float32), y), num_parallel_calls=AUTO)
  dataset = dataset.map(lambda x,y: (tf.keras.layers.Rescaling(scale=1./255)(x), y), num_parallel_calls=AUTO)

  return dataset

training_config = {
"ENABLE_DETERMINISM": False,
"SEED": 17,
"AUTO": tf.data.AUTOTUNE,
"TRAIN_BATCH_SIZE": 64,
"TEST_BATCH_SIZE": 64,
"NUMBER_OF_EPOCHS": 500,
"EARLY_STOPPING_TOLERANCE": 10,
"LEARNING_RATE": 1e-4,
"IMAGE_SIZE": 128,
"SHUFFLE_BUFFER": 4096,
"TPU": False,
"AUGMENTATIONS_ROTATE": False,
"AUGMENTATIONS_FLIP_HORIZONTALLY": False,
"AUGMENTATIONS_FLIP_VERTICALLY": False,
"AUGMENTATIONS_ZOOM": False,
"AUGMENTATIONS_RANDOM_NOISE": False,
"AUGMENTATIONS_CENTER_NOISE": False,
"AUGMENTATIONS_OUTSIDE_NOISE": False,
"AUGMENTATIONS_PERLIN_CENTER_NOISE": False,
"USE_ADABELIEF_OPTIMIZER": False,
"GALAXY_DATASET": "SDSS",
"TRAIN_DATASET_SIZE": 2 * 4096 + 2048,
"TEST_DATASET_SIZE": 1025,
"STRETCH": False,
"STEPS_PER_EXECUTION": 1,
"FOLDS": 1,
"NUMBER_OF_CLASSES": 11
}
training_config["WANDB_PROJECT_NAME"] = f"TrainTestValidation, {training_config['NUMBER_OF_CLASSES']} classes"
training_config["EXPERIMENT_DESCRIPTION"] = f"asd"
training_config["LOCAL_GCP_PATH_BASE"] = f"TFDataset/11Classes"
training_config["REMOTE_GCP_PATH_BASE"] = f"TFDataset/11Classes"
training_config["DATASET_PATH"] = f"tuning"

class SparseF1Score(tf.keras.metrics.F1Score):

    def __init__(self, name='f1_score', **kwargs):
        super().__init__(name=name, **kwargs)

    def update_state(self, y_true, y_pred, sample_weight=None):
        super().update_state(tf.reshape(tf.one_hot(y_true, training_config["NUMBER_OF_CLASSES"]), [-1, training_config["NUMBER_OF_CLASSES"]]), y_pred, sample_weight)

    def result(self):
        return super().result()

@keras.saving.register_keras_serializable()
class FocalConv(keras.layers.Layer):
    def __init__(self, filters, kernel_size, factor, activation):
        super().__init__()
        self.filters = filters
        self.kernel = kernel_size
        self.activation = activation
        self.factor = factor
        self.conv1 = Conv2D(filters=self.filters, kernel_size=self.kernel, activation=self.activation, padding='same')

    def build(self, input_shape):
        self.gauss = self.make_gaussian(size=input_shape[1], depth=self.filters, factor=self.factor)

    def call(self, x, training=False):
      x = self.conv1(x, training=training)
      batch_size = tf.shape(x)[0]
      xx = tf.repeat(self.gauss, repeats=batch_size, axis=0)
      x = multiply([x, xx])

      return x
    
    def get_config(self):
       return {
          "filters": self.filters,
          "kernel_size": self.kernel,
          "activation": self.activation,
          "factor": self.factor
       }

    def make_gaussian(self, size: int, depth: int, factor: float):
        """ Make a square gaussian kernel.

        size is the length of a side of the square
        fwhm is full-width-half-maximum, which
        can be thought of as an effective radius.
        """
        fwhm = size // 2
        x = np.arange(0, size, 1, float)
        y = x[:,np.newaxis]

        x0 = y0 = size // 2

        gauss = np.exp(-1 * factor * np.log(2) * ((x-x0) ** 2 + (y-y0) ** 2) / fwhm **2)

        return tf.expand_dims(np.stack([gauss for _ in range(depth)], -1), axis=0)


@keras.saving.register_keras_serializable()
class WaveConv(keras.layers.Layer):
    def __init__(self, filters, kernel_size, k_factor, activation):
        super().__init__()
        self.filters = filters
        self.kernel = kernel_size
        self.activation = activation
        self.k_factor = k_factor
        self.conv1 = Conv2D(filters=self.filters, kernel_size=self.kernel, activation=self.activation, padding='same')

    def build(self, input_shape):
        self.wave = self.make_wave(size=input_shape[1], depth=self.filters, k_factor=self.k_factor)

    def call(self, x, training=False):
      x = self.conv1(x, training=training)
      batch_size = tf.shape(x)[0]
      xx = tf.repeat(self.wave, repeats=batch_size, axis=0)
      x = multiply([x, xx])

      return x
    
    def get_config(self):
       return {
          "filters": self.filters,
          "kernel_size": self.kernel,
          "activation": self.activation,
          "k_factor": self.k_factor
       }

    def make_wave(self, size: int, depth: int, k_factor: float = 2):
        x = np.arange(0, size, 1, float)
        y = x[:,np.newaxis]
        x0 = y0 = size // 2

        k = np.pi / k_factor
        x, y = x - x0, y - y0
        l = np.sqrt(x * x + y * y)
        u = np.cos(k * l )
        u = normalize(u)
        return 1 - tf.expand_dims(np.stack([u for _ in range(depth)], -1), axis=0)









@keras.saving.register_keras_serializable()
class FocalLayer(keras.layers.Layer):
    def __init__(self, filters, factor):
        super().__init__()
        self.filters = filters
        self.factor = factor

    def build(self, input_shape):
        self.gauss = self.make_gaussian(size=input_shape[1], depth=self.filters, factor=self.factor)

    def call(self, x, training=False):
      batch_size = tf.shape(x)[0]
      xx = tf.repeat(self.gauss, repeats=batch_size, axis=0)
      x = multiply([x, xx])

      return x
    
    def get_config(self):
       return {
          "filters": self.filters,
          "factor": self.factor
       }

    def make_gaussian(self, size: int, depth: int, factor: float):
        """ Make a square gaussian kernel.

        size is the length of a side of the square
        fwhm is full-width-half-maximum, which
        can be thought of as an effective radius.
        """
        fwhm = size // 2
        x = np.arange(0, size, 1, float)
        y = x[:,np.newaxis]

        x0 = y0 = size // 2

        gauss = np.exp(-1 * factor * np.log(2) * ((x-x0) ** 2 + (y-y0) ** 2) / fwhm **2)

        return tf.expand_dims(np.stack([gauss for _ in range(depth)], -1), axis=0)


@keras.saving.register_keras_serializable()
class WaveLayer(keras.layers.Layer):
    def __init__(self, filters, k_factor):
        super().__init__()
        self.filters = filters
        self.k_factor = k_factor

    def build(self, input_shape):
        self.wave = self.make_wave(size=input_shape[1], depth=self.filters, k_factor=self.k_factor)

    def call(self, x, training=False):
      batch_size = tf.shape(x)[0]
      xx = tf.repeat(self.wave, repeats=batch_size, axis=0)
      x = multiply([x, xx])

      return x
    
    def get_config(self):
       return {
          "filters": self.filters,
          "k_factor": self.k_factor
       }

    def make_wave(self, size: int, depth: int, k_factor: float = 2):
        x = np.arange(0, size, 1, float)
        y = x[:,np.newaxis]
        x0 = y0 = size // 2

        k = np.pi / k_factor
        x, y = x - x0, y - y0
        l = np.sqrt(x * x + y * y)
        u = np.cos(k * l )
        u = normalize(u)
        return 1 - tf.expand_dims(np.stack([u for _ in range(depth)], -1), axis=0)


@keras.saving.register_keras_serializable()
class BarLayer(keras.layers.Layer):
    def __init__(self, filters, angle, alpha):
        super().__init__()
        self.filters = filters
        self.angle = angle
        self.alpha = alpha

    def build(self, input_shape):
        self.bar = self.make_bar(input_shape[1], self.alpha, self.angle, self.filters)

    def call(self, x, training=False):
      batch_size = tf.shape(x)[0]
      xx = tf.repeat(self.bar, repeats=batch_size, axis=0)
      x = multiply([x, xx])

      return x
    
    def get_config(self):
       return {
          "filters": self.filters,
          "alpha": self.alpha,
          "angle": self.angle
       }

    def make_bar(self, size: int, alpha: int, angle: int, depth: int):
        arr = np.zeros((size, size))

        bar_size = size // 10
        padding = (size - bar_size) // 2

        for i in range(padding):
            arr[i, :] = ((float(i) / padding) ** alpha)

        arr[padding: padding+bar_size, :] = 1

        for j in range(padding):
            arr[padding + bar_size + j, :] = (-1 * (float(j) / padding) + 1) ** alpha

        arr = rotate(arr, angle, reshape = False, mode='nearest')   
        return tf.expand_dims(np.stack([arr for _ in range(depth)], axis = -1), axis=0)



def model_builder(hp):
  
  inputs = Input(shape=(training_config["IMAGE_SIZE"], training_config["IMAGE_SIZE"], 3))

  focal_1_o  =  FocalLayer(filters=3, factor = 1)(inputs)
  focal_3_o  =  FocalLayer(filters=3, factor = 3)(inputs)
  focal_5_o  =  FocalLayer(filters=3, factor = 5)(inputs)
  wave1_o  =  WaveLayer(filters=3, k_factor = 1)(inputs)
  wave3_o  =  WaveLayer(filters=3, k_factor = 3)(inputs)
  wave5_o  =  WaveLayer(filters=3, k_factor = 5)(inputs)
  bar0_o   = BarLayer(alpha = 7, angle = 0,  filters = 3)(inputs)
  bar45_o  = BarLayer(alpha = 7, angle = 45,  filters = 3)(inputs)
  bar90_o  = BarLayer(alpha = 7, angle = 90,  filters = 3)(inputs)
  bar135_o = BarLayer(alpha = 7, angle = 135,  filters = 3)(inputs)

  x = Concatenate()([focal_1_o, focal_3_o, focal_5_o, wave1_o, wave3_o, wave5_o, bar0_o, bar45_o, bar90_o, bar135_o])
  # x = AveragePooling2D(2)(x)

  num_layers = hp.Int('num_layers', min_value=3, max_value=5, step=1)
  for i in range(num_layers):

    layer_type = hp.Choice(f"type_{i}", ["SimpleConv", "FocalConv"])
    focal_factor = hp.Int(f'focal_factor_{i}', parent_name=f'type_{i}', parent_values=["FocalConv"], min_value=1, max_value=10, step=1)
    kernel_size = hp.Int(f'kernel_size_{i}', min_value=3, max_value=7, step=2)
    filters = hp.Int(f'filters_{i}', min_value=16, max_value=64, step=16)

    if (layer_type == "SimpleConv"):
      x = Conv2D(filters=filters, kernel_size=kernel_size, activation='relu', padding='same')(x)
    elif(layer_type == "FocalConv"):
      x = FocalConv(filters=filters, kernel_size=kernel_size, activation='relu', factor= focal_factor)(x)

    x = BatchNormalization()(x)
    x = MaxPool2D(2)(x)


  x = Flatten()(x)
  x = Dropout(0.5)(x)

  num_dense = hp.Int('num_dense', min_value=1, max_value=3, step=1)
  for j in range(num_dense):
    dense_output = hp.Int(f'dense_output_{j}', min_value=32, max_value=256, step=32)
    x = Dense(dense_output, activation='relu')(x)

  outputs = Dense(training_config["NUMBER_OF_CLASSES"], activation='softmax')(x)

  model = tf.keras.Model(inputs=inputs, outputs=outputs)
  
  model.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-4, beta_1=0.9, beta_2=0.999, epsilon=1e-08),
              loss=keras.losses.SparseCategoricalCrossentropy(),
              metrics=[tf.keras.metrics.SparseCategoricalAccuracy(), SparseF1Score(
                average='macro',
            )])

  return model


@keras.saving.register_keras_serializable(package="mlp", name="mlp")
def mlp(x, hidden_units, dropout_rate):
    for units in hidden_units:
        x = layers.Dense(units, activation=keras.activations.gelu)(x)
        x = layers.Dropout(dropout_rate)(x)
    return x

@keras.saving.register_keras_serializable(package="PatchEncoder")
class PatchEncoder(layers.Layer):
    def __init__(self, num_patches, projection_dim):
        super().__init__()
        self.num_patches = num_patches
        self.projection_dim = projection_dim
        self.projection = layers.Dense(units=self.projection_dim)
        self.position_embedding = layers.Embedding(
            input_dim=self.num_patches, output_dim=self.projection_dim
        )

    def call(self, patch):
        positions = ops.expand_dims(
            ops.arange(start=0, stop=self.num_patches, step=1), axis=0
        )
        projected_patches = self.projection(patch)
        encoded = projected_patches + self.position_embedding(positions)
        return encoded

    def get_config(self):
        return {"num_patches": self.num_patches, "projection_dim": self.projection_dim}

@keras.saving.register_keras_serializable(package="REshape")
class Reshape(layers.Layer):
    def __init__(self):
        super().__init__()

    def call(self, patch):
      input_shape = ops.shape(patch)
      batch_size = input_shape[0]
      return ops.reshape(
          patch,
          (
              batch_size,
              # num_patches_h * num_patches_w,
              11,
              128 * 128 * 3,
          ),
      )

    def get_config(self):
        return {"num_patches": self.num_patches, "projection_dim": self.projection_dim}

def Paper3TRansformerV3(hp):
    patch_size = 128
    projection_dim = hp.Int('projection_dim', min_value=8, max_value=64, step=8)
    transformer_layers = hp.Int('layers', min_value=1, max_value=4, step=1)
    num_heads = hp.Int('heads', min_value=2, max_value=5, step=1)
    head_size = hp.Int('head_size', min_value=128, max_value=1024, step=128)
    dropout_rate = 0.5

    transformer_units = [
        projection_dim * 2,
        projection_dim,
    ]
    mlp_head_units = [
        head_size * 2,
        head_size,
    ]


    inputs = keras.Input(shape=(training_config["IMAGE_SIZE"], training_config["IMAGE_SIZE"], 3))
    

    focal1_o  =  FocalLayer(filters=3, factor = 1)(inputs)
    focal3_o  =  FocalLayer(filters=3, factor = 3)(inputs)
    focal5_o  =  FocalLayer(filters=3, factor = 5)(inputs)
    wave1_o  =  WaveLayer(filters=3, k_factor = 1)(inputs)
    wave3_o  =  WaveLayer(filters=3, k_factor = 3)(inputs)
    wave5_o  =  WaveLayer(filters=3, k_factor = 5)(inputs)
    bar0_o   = BarLayer(alpha = 7, angle = 0,  filters = 3)(inputs)
    bar45_o  = BarLayer(alpha = 7, angle = 45,  filters = 3)(inputs)
    bar90_o  = BarLayer(alpha = 7, angle = 90,  filters = 3)(inputs)
    bar135_o = BarLayer(alpha = 7, angle = 135,  filters = 3)(inputs)

    patches = Concatenate()([inputs, focal1_o, focal3_o, focal5_o,  wave1_o, wave3_o, wave5_o, bar0_o, bar45_o, bar90_o, bar135_o])
    # patches = AveragePooling2D(2)(patches)
    patches = Reshape()(patches)

    num_patches = 11
    # Create patches.
    # Encode patches.
    encoded_patches = PatchEncoder(num_patches, projection_dim)(patches)

    # Create multiple layers of the Transformer block.
    for _ in range(transformer_layers):
        # Layer normalization 1.
        x1 = layers.LayerNormalization(epsilon=1e-6)(encoded_patches)
        # Create a multi-head attention layer.
        attention_output = layers.MultiHeadAttention(
            num_heads=num_heads, key_dim=projection_dim, dropout=dropout_rate
        )(x1, x1)
        # Skip connection 1.
        x2 = layers.Add()([attention_output, encoded_patches])
        # Layer normalization 2.
        x3 = layers.LayerNormalization(epsilon=1e-6)(x2)
        # MLP.
        x3 = mlp(x3, hidden_units=transformer_units, dropout_rate=dropout_rate)
        # Skip connection 2.
        encoded_patches = layers.Add()([x3, x2])

    # Create a [batch_size, projection_dim] tensor.
    representation = layers.LayerNormalization(epsilon=1e-6)(encoded_patches)
    representation = layers.Flatten()(representation)
    representation = layers.Dropout(dropout_rate)(representation)
    # Add MLP.
    features = mlp(representation, hidden_units=mlp_head_units, dropout_rate=dropout_rate)
    # Classify outputs.
    logits = layers.Dense(training_config["NUMBER_OF_CLASSES"], activation='softmax')(features)
    # Create the Keras model.
    model = keras.Model(inputs=inputs, outputs=logits)

    model.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-4, beta_1=0.9, beta_2=0.999, epsilon=1e-08),
              loss=keras.losses.SparseCategoricalCrossentropy(),
              metrics=[tf.keras.metrics.SparseCategoricalAccuracy(), SparseF1Score(
                average='macro',
            )])

    return model


def dense_layer(x, dense_1, dense_2, output_size, average_pool):
  x = AveragePooling2D(average_pool)(x)
  x = Dense(dense_1, activation='relu')(x)
  x = Dropout(0.5)(x)
  x = Dense(dense_2, activation='relu')(x)
  x = Dropout(0.5)(x)
  x = Dense(output_size, activation='relu')(x)

  return x

def Paper3ConvDenseV2(hp):
  
  dense_1 = hp.Int(f'dense_1', min_value=32, max_value=92, step=32)
  dense_2 = hp.Int(f'dense_2', min_value=32, max_value=256, step=32)
  output_size = hp.Int(f'output', min_value=8, max_value=64, step=8)
  pooling = hp.Int(f'pooling', min_value=2, max_value=4, step=2)

  inputs = Input(shape=(training_config["IMAGE_SIZE"], training_config["IMAGE_SIZE"], 3))

  focal_1_o  =  FocalLayer(filters=3, factor = 1)(inputs)
  focal_1_o  =  dense_layer(focal_1_o, dense_1, dense_2, output_size, pooling)


  focal_3_o  =  FocalLayer(filters=3, factor = 3)(inputs)
  focal_3_o  =  dense_layer(focal_3_o, dense_1, dense_2, output_size, pooling)

  focal_5_o  =  FocalLayer(filters=3, factor = 5)(inputs)
  focal_5_o  =  dense_layer(focal_5_o, dense_1, dense_2, output_size, pooling)

  wave1_o  =  WaveLayer(filters=3, k_factor = 1)(inputs)
  wave1_o  =  dense_layer(wave1_o, dense_1, dense_2, output_size, pooling)

  wave3_o  =  WaveLayer(filters=3, k_factor = 3)(inputs)
  wave3_o  =  dense_layer(wave3_o, dense_1, dense_2, output_size, pooling)

  wave5_o  =  WaveLayer(filters=3, k_factor = 5)(inputs)
  wave5_o  =  dense_layer(wave5_o, dense_1, dense_2, output_size, pooling)

  bar0_o   = BarLayer(alpha = 7, angle = 0,  filters = 3)(inputs)
  bar0_o  =  dense_layer(bar0_o, dense_1, dense_2, output_size, pooling)

  bar45_o  = BarLayer(alpha = 7, angle = 45,  filters = 3)(inputs)
  bar45_o  =  dense_layer(bar45_o, dense_1, dense_2, output_size, pooling)

  bar90_o  = BarLayer(alpha = 7, angle = 90,  filters = 3)(inputs)
  bar90_o  =  dense_layer(bar90_o, dense_1, dense_2, output_size, pooling)

  bar135_o = BarLayer(alpha = 7, angle = 135,  filters = 3)(inputs)
  bar135_o  =  dense_layer(bar135_o, dense_1, dense_2, output_size, pooling)

  x = Concatenate()([focal_1_o, focal_3_o, focal_5_o, wave1_o, wave3_o, wave5_o, bar0_o, bar45_o, bar90_o, bar135_o])
  # x = AveragePooling2D(2)(x)
  
  x = BatchNormalization()(x)
  x = Flatten()(x)

  num_dense = hp.Int('num_dense', min_value=1, max_value=3, step=1)
  for j in range(num_dense):
    dense_output = hp.Int(f'dense_output_{j}', min_value=32, max_value=256, step=32)
    x = Dense(dense_output, activation='relu')(x)

  outputs = Dense(training_config["NUMBER_OF_CLASSES"], activation='softmax')(x)

  model = tf.keras.Model(inputs=inputs, outputs=outputs)
  
  model.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-4, beta_1=0.9, beta_2=0.999, epsilon=1e-08),
            loss=keras.losses.SparseCategoricalCrossentropy(),
            metrics=[tf.keras.metrics.SparseCategoricalAccuracy(), SparseF1Score(
            average='macro',
        )])

  return model


gpus = tf.config.list_physical_devices('GPU')
strategy = tf.distribute.MirroredStrategy()

test_dataset = get_intial_fold_dataset(
    training_config,
    f"{training_config['REMOTE_GCP_PATH_BASE']}/{training_config['DATASET_PATH']}/test",
    seed = training_config["SEED"],
    shuffle = False,
    cache = True).batch(training_config["TEST_BATCH_SIZE"])

train_dataset = get_intial_fold_dataset(
    training_config,
    f"{training_config['REMOTE_GCP_PATH_BASE']}/{training_config['DATASET_PATH']}/train",
    training_config["SEED"],
    shuffle = True,
    cache = True).batch(training_config["TEST_BATCH_SIZE"])

stop_early = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=10)

# train_dataset = train_dataset.repeat()
# test_dataset = test_dataset.repeat()

tuner = kt.Hyperband(
    Paper3ConvDenseV2,
    objective=kt.Objective("val_loss", direction="min"),
    max_epochs=100,
    overwrite=False,
    hyperband_iterations = 5,
    directory='tuning_models',
    project_name='paper3_parallel_dense_best_loss',
    max_consecutive_failed_trials=1,
    distribution_strategy=strategy)

tuner.search(x= train_dataset,
            validation_data = test_dataset,
            epochs = 500,
            callbacks=[stop_early],
            # steps_per_epoch = (5 * 4096) // 128,
            # validation_steps = 2264 // 128,
            verbose = 1,
            shuffle = False)